# 第三课：AI评估入门 —— 如何判断AI回答好不好

## 学习目标
- 理解为什么评估 AI 输出比评估传统软件难得多
- 设计简单的评分标准，手工评估 AI 的回答
- 使用「AI 做裁判」来评估 AI
- 批判性地审视 AI 输出，培养评估思维

> 评估是 AI 工程化中最重要也最常被忽视的环节。没有评估，你就不知道 AI 是在进步还是在退步。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：自己当「AI 裁判」——手工评估 AI 回答

### 活动目标
先让 AI 回答几个问题，然后你自己设计评分标准，逐条打分。建立「什么是好回答」的判断力。

In [ ]:
# 三个不同领域的问题
questions = [
    {'category': '常识', 'question': '请解释温室效应的基本原理，以及它对地球气候的影响。'},
    {'category': '创意', 'question': '请写一段200字的科幻小说开头，主题是最后一个人类与AI的对话。'},
    {'category': '逻辑', 'question': '如果所有的猫都怕水，小明家的宠物不怕水，那么小明家的宠物是猫吗？请分析推理过程。'}
]

answers = {}
for i, q in enumerate(questions):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'user','content':q['question']}],
        temperature=0.5)
    answers[i] = r.choices[0].message.content
    print(f'\n问题{i+1}【{q["category"]}】')
    print(f'问题：{q["question"]}')
    print(f'AI回答：\n{answers[i]}')
    print()

### 手工评估任务

现在请你当「AI 裁判」。按以下标准逐项打分（每项 1-5 分）：

| 维度 | 1分 | 3分 | 5分 |
|------|-----|-----|-----|
| 准确性 | 有事实错误 | 基本正确，有小瑕疵 | 完全准确 |
| 完整性 | 遗漏关键信息 | 覆盖大部分要点 | 全面深入 |
| 清晰度 | 混乱难懂 | 基本通顺 | 结构清晰 |
| 有用性 | 毫无帮助 | 有一定帮助 | 直接解决问题 |

在下面的代码单元中填入你的评分：

In [ ]:
# 请在这里填入你的评分（每项 1-5 分）
# 四个维度：准确性、完整性、清晰度、有用性
my_scores = {
    1: [4, 4, 5, 4],  # 问题1：请根据实际情况修改
    2: [3, 4, 4, 3],  # 问题2
    3: [5, 3, 4, 4],  # 问题3
}

print('你的评分结果：')
print(f'{"问题":<10}{"准确性":<10}{"完整性":<10}{"清晰度":<10}{"有用性":<10}{"总分":<8}')
print('-' * 58)
for q_num, scores in my_scores.items():
    total = sum(scores)
    print(f'{"问题" + str(q_num):<10}{scores[0]:<10}{scores[1]:<10}{scores[2]:<10}{scores[3]:<10}{total:<8}')
best = max(my_scores, key=lambda k: sum(my_scores[k]))
print(f'\n得分最高：问题{best}，总分{sum(my_scores[best])}')

### 讨论
- 给 AI 回答打分时，最难判断的是哪个维度？
- 如果你的评分和旁边同学不同，差异在哪里？
- 好的评估标准应该具备什么特征？

---

## 活动二：让 AI 当裁判 ——「AI as a Judge」

### 活动目标
让 AI 自己来评估 AI 的回答。给 AI 一套评分标准，让它打分。然后对比 AI 的评分和你自己的评分。

In [ ]:
# 让 AI 当裁判评估
eval_prompt = '你是一位严格的AI输出评估专家。请按照以下标准对AI回答评分。\n'
'评分标准（每项1-5分）：\n'
'1. 准确性：1=有明显错误，5=完全准确\n'
'2. 完整性：1=严重遗漏，5=全面深入\n'
'3. 清晰度：1=混乱难懂，5=清晰严谨\n'
'4. 有用性：1=毫无帮助，5=非常有帮助\n'
'请对每个回答逐项给出评分。'

for i, q in enumerate(questions):
    print(f'\n=== AI裁判评估：问题{i+1}【{q["category"]}】===')
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':'system','content':'你是一位严格的AI输出评估专家。'},
            {'role':'user','content':f'{eval_prompt}\n\n问题：{q["question"]}\n\nAI回答：{answers[i]}'}
        ],
        temperature=0.2)
    print(r.choices[0].message.content)

### 讨论
- AI 裁判的评分和你的评分一致吗？
- AI 裁判会不会「心软」（倾向于给高分）？
- 如果修改评分标准中的措辞，AI 的评分会不会变化？

---

## 活动三：A/B 对比——两个 AI 谁写得更好？

### 活动目标
不看分数，只比谁更好。把两个回答放在一起直接比较——这是最直观的评估方式。

In [ ]:
# A/B 盲评对比
topic = 'AI 对教育的影响'

# 回答 A：学术风格
print('【回答 A：学术风格】')
ra = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是教育学研究学者，用严谨学术风格回答。'},
        {'role':'user','content':f'请写一篇200字短文，主题是{topic}。'}
    ], temperature=0.3)
text_a = ra.choices[0].message.content
print(text_a)

# 回答 B：通俗风格
print('\n【回答 B：通俗风格】')
rb = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role':'system','content':'你是科普博主，用通俗易懂、生动有趣的方式回答。'},
        {'role':'user','content':f'请写一篇200字短文，主题是{topic}。'}
    ], temperature=0.7)
text_b = rb.choices[0].message.content
print(text_b)

# AI 盲评
print('\n【AI 盲评】')
rj = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'对比以下两篇短文，判断哪篇更好并说明理由。\n\n短文A：{text_a}\n\n短文B：{text_b}'}],
    temperature=0.2)
print(rj.choices[0].message.content)

### 讨论
- 你同意 AI 裁判的盲评结果吗？
- 学术风格 vs 通俗风格，哪种更适合「教育」这个主题？
- 如果你来选一个内容创作助手，你会选哪种风格的模型？

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| 手工评估 | 设计评分标准，亲自给 AI 回答打分 |
| AI 做裁判 | 用 AI 来评估 AI，理解其优势与局限 |
| 评分标准设计 | 体会标准措辞如何影响评估结果 |
| A/B 对比 | 直接比较两个回答，判断优劣 |

### 课后练习
1. 找5个不同领域的 AI 回答，用同样的标准打分
2. 设计一套你自己的评估标准，包含至少5个维度
3. 思考：如果让你设计一个自动评估系统，你会怎么设计？